# HW8 — Градиентный бустинг, MLflow-трекинг и честный лидерборд

> **Начинающему трейдеру — начать отсюда.** Раздел «Справочник» объясняет простыми словами
> бустинг, MLflow и все метрики, которые используются дальше. Основной текст показывает, как
> строится таблица результатов, которой нельзя соврать.

## Справочник для начинающего алгоритмического трейдинга

### Как устроен градиентный бустинг

Общая идея всех четырёх библиотек одна и та же: **строить много слабых моделей последовательно,
где каждая следующая исправляет ошибки предыдущих.** Шаг за шагом ансамбль «догоняет» то, что
не смогла ухватить одна модель. Отличаются только детали:

| Реализация | Отличие простыми словами |
|---|---|
| **LightGBM** | Бьёт данные по «листьям» (наиболее разделяющим признакам) — быстрее всего на табличных данных |
| **XGBoost** | Классическая реализация с упором на регуляризацию и устойчивость |
| **CatBoost** | Всстроенная обработка категориальных признаков, меньше ручной настройки |
| **HistGradientBoosting** | Реализация scikit-learn на гистограммах — удобна, если хочется остаться в экосистеме sklearn |
| **Logistic regression** | Линейный контроль: самый дешёвый способ проверить, делают ли бустеры вообще что-нибудь |

### Валидация и защита от обмана

| Термин | Что это простыми словами |
|---|---|
| **Walk-forward** | Учимся только на прошлом, проверяем на следующем куске, двигаем окно вправо — имитация реальной торговли |
| **Purged K-fold + embargo** | K-fold, из которого выброшены строки с меткой, пересекающей тест, плюс карантинная полоса вокруг |
| **OOS (out-of-sample)** | Данные, которые модель при обучении не видела |
| **Climatology** | Точность классификатора «всегда предсказывать самый частый класс» — пол для классификатора |
| **Buy-and-hold** | Sharpe и CAGR простого удержания индекса — пол для торговой стратегии |
| **Transaction costs (издержки)** | Комиссия и проскальзывание. Указываются в базовых пунктах (bp): 10 bp = 0.1% |
| **Permutation importance** | Перемешиваем столбец в **тестовых** данных и смотрим, насколько ухудшилась модель. Показывает, чем модель реально пользуется |
| **Calibration / ECE / Brier** | Насколько заявленные вероятности совпадают с реальной частотой событий |
| **Reliability diagram** | График «предсказанная вероятность против фактической частоты» по корзинам |

### Отчётность и неопределённость

| Термин | Что это простыми словами |
|---|---|
| **MLflow** | Система журнала экспериментов: параметры, метрики, теги — всё сохраняется в базу, а не «в голове» |
| **PSR** | Вероятность того, что Sharpe статистически отличается от нуля |
| **DSR** | То же, но против Sharpe *лучшей из N попыток*, полученной на удачу — учитывает сам факт перебора |
| **MinTRL** | Сколько лет нужно вести трек-рекорд, чтобы подтвердить заявленный Sharpe |
| **Block bootstrap** | Ресемплирование блоками, а не по одному наблюдению — сохраняет серийную зависимость |

## Зачем существует этот ноутбук

Это **единственный** ноутбук репозитория, который начинается с модели побольше, и это
намеренно: после того как `HW6` показал, что зооподбор статистических моделей не бьёт константу,
а `HW7` показал, что перебор из 24 конфигураций градиентного бустинга не бьёт «просто держи
SPY», интересен уже не вопрос *может ли бустинг обойти бенчмарк*. Интересен **как выглядит
таблица бустинга, когда она построена так, что не может соврать.**

Сравнение бустеров легко подделать и трудно сделать честно:

| Режим провала | Как это выглядит | Защита здесь |
|---|---|---|
| Выбор лучшего из множества | сообщить победителя перебора как *тот самый* результат | число попыток передаётся в DSR; печатаются **и** максимум, **и** распределение |
| Ранжирование только по точности | точность топ-1 подаётся как результат | точность паруется с точностью большинства и с чистым Sharpe |
| Важность признаков по обучающим данным | важность, просто воспроизводящая метки | **пермутационная** важность считается **вне выборки** |
| Нет транзакционных издержек | валовой Sharpe, исчезающий на 10 bp | издержки внутри потока доходностей |
| Отсутствует бенчмарк | «91% точности!» без бенчмарка | climatology и buy-and-hold считаются **сначала** |
| Воспроизводимость на словах | результаты цитируются по памяти | каждый запуск логируется в MLflow и читается обратно из хранилища |

## Что производит этот ноутбук

1. **Зооподбор моделей** — LightGBM, XGBoost, CatBoost, HistGradientBoosting из scikit-learn и
   логистический бейзлайн — все на одном протоколе walk-forward с очищенными фолдами, одним
   общим конвейером предобработки и одинаковыми допущениями об издержках.
2. **Хранилище MLflow на диске** с параметрами, метриками и тегами по каждой модели и
   лидербордом, **прочитанным обратно из хранилища**, а не из переменных в памяти — чтобы
   трекинг действительно испытывался, а не просто назывался.
3. **Пермутационную важность вне выборки** — что модель использует, а не что видела при обучении.
4. **Калибровку вероятностей** (reliability, ECE, Brier skill score) для победителя — именно она
   определяет, пригодны ли его сигналы для размера позиции.
5. Честный **вердикт**, следующий из запуска, включая возможность — установленную заранее, а не
   надеянную — что лучший бустер всё равно не обходит свои бенчмарки.

In [1]:
import json
import os
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from mlfin_core import (
    TRADING_DAYS,
    FoldSafePreprocessor,
    annualised_return,
    apply_causal_execution,
    bootstrap_sharpe_ci,
    buy_and_hold_returns,
    deflated_sharpe_ratio,
    load_ohlcv,
    make_direction_target,
    make_feature_frame,
    min_track_record_length,
    positions_to_returns,
    probabilistic_sharpe_ratio,
    profit_factor,
    purged_kfold_splits,
    sharpe_ratio,
    sortino_ratio,
    win_rate,
)

print("mlfin_core загружен | TRADING_DAYS =", TRADING_DAYS)

mlfin_core загружен | TRADING_DAYS = 252


## Конфигурация

**Протокол намеренно идентичен `HW7`** — тот же тикер, те же признаки, та же цель, та же точка
разбиения, те же издержки, тот же embargo. Удержать всё фиксированным, кроме самого алгоритма —
единственный способ, чтобы A/B-сравнение LightGBM и CatBoost что-нибудь значило; это также
означает, что числа здесь можно напрямую сверить с `HW7`.

Конфигурация, выбранная `HW7` на кросс-валидации, переиспользуется **без изменений**. Это не
удобство: отдельный тюнинг для каждой модели означал бы, что каждый бустер получает разное
число шансов выглядеть хорошо, и лидерборд мерил бы усилия по тюнингу, а не качество модели.

In [2]:
TICKER = "SPY"
START = "2015-01-01"
DATA_END = "2026-10-05"   # зафиксировано: опубликованные числа обязаны воспроизводиться
DATA_DIR = os.path.join(NOTEBOOK_DIR, "data")   # закоммиченный снимок: читается вместо запроса к API
HORIZON = 1
DEAD_BAND = 0.0005
EMBARGO = 5
OOS_FRACTION = 0.20
REFIT_EVERY = 21
COST_BPS = 10.0
SR_TARGET = 1.0
RNG_SEED = 0

# Конфигурация, отобранная HW7 очищенной кросс-валидацией. Переиспользуется, а не пере-тюнится.
BASE_CONFIG = {
    "learning_rate": 0.03,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 0.0,
    "max_iter": 150,
}
N_BOOSTERS = 4          # сколько конфигураций бустинга будет проверено; подаётся в DSR

df, source = load_ohlcv(TICKER, START, end=DATA_END, local_dir=DATA_DIR, allow_synthetic=True)
print(f"{TICKER}: {len(df)} баров | {df.index[0].date()} -> {df.index[-1].date()}")
print(f"источник данных: {source}")
if "synthetic" in str(source).lower():
    print("ВНИМАНИЕ: используется синтетическая заглушка. Рейтинги ниже описывают генератор,")
    print("а не рынок, и их нельзя цитировать как результат по рынку.")
print(f"\nконфигурация, переиспользованная из HW7: {BASE_CONFIG}")

SPY: 3066 баров | 2015-01-02 -> 2026-10-02
источник данных: SPY (local: C:\Users\MV\AppData\Local\Temp\opencode\ML_Fin_Notebooks\data\SPY.csv)

конфигурация, переиспользованная из HW7: {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_iter': 150}


## Признаки, цель и два бенчмарка

Конструкция та же, что в `HW7`. **Два нулевых бенчмарка считаются до того, как обучена
хотя бы одна модель** — намеренно, чтобы их нельзя было тихо переопределить позже:

* **Climatology** — точность классификатора, который всегда предсказывает класс большинства
  обучающей выборки. Это порог, который обязан пройти классификатор направления, чтобы что-либо
  доказать.
* **Buy-and-hold** — Sharpe и CAGR удержания SPY на идентичных барах, без вычетов. Это порог,
  который должна пройти long/flat стратегия, чтобы её стоило разворачивать.

**Любое утверждение «модель достигает X» без одного из этих двух чисел рядом не оценивалось.**

In [3]:
feats = make_feature_frame(df, n_lags=5, extra_indicators=True)
y = make_direction_target(df["Close"], horizon=HORIZON, deadband=DEAD_BAND, labels=(0, 1, 2))
y = y.rename("y")

data = feats.join(y, how="inner")
data = data[data["y"].notna()].copy()
close = df["Close"].reindex(data.index)

n_obs = len(data)
n_oos = int(round(n_obs * OOS_FRACTION))
X = data.drop(columns=["y"])
y = data["y"]
X_IS, y_IS = X.iloc[: n_obs - n_oos], y.iloc[: n_obs - n_oos]
X_OOS, y_OOS = X.iloc[n_obs - n_oos:], y.iloc[n_obs - n_oos:]
close_all = close.to_numpy()

CLASS_NAMES = {0: "sell", 1: "buy", 2: "hold"}
CLASS_ORDER = [0, 1, 2]
BUY_I, SELL_I = 1, 0

print(f"признаки            : {X.shape[1]} столбцов")
print(f"IS (только отбор)   : {len(X_IS)} баров  {X_IS.index[0].date()} -> {X_IS.index[-1].date()}")
print(f"OOS (финальное окно): {len(X_OOS)} баров  {X_OOS.index[0].date()} -> {X_OOS.index[-1].date()}")

MAJORITY_CLASS = int(y_IS.value_counts().idxmax())
CLIMATOLOGY_ACC = float((y_OOS == MAJORITY_CLASS).mean())
print(f"\nточность климатологии (всегда '{CLASS_NAMES[MAJORITY_CLASS]}'): {CLIMATOLOGY_ACC:.4f}")

bh_full = buy_and_hold_returns(close)
BH_SHARPE_FULL = sharpe_ratio(bh_full)
BH_CAGR_FULL = annualised_return(bh_full)
print(f"Sharpe buy-and-hold  : {BH_SHARPE_FULL:.3f}")
print(f"CAGR buy-and-hold    : {BH_CAGR_FULL:+.4f}")
print("\nЭти две строки — нулевая строка лидерборда. Всё ниже измеряется относительно них.")

признаки            : 23 столбцов
IS (только отбор)   : 2452 баров  2015-01-02 -> 2024-05-27
OOS (финальное окно): 613 баров  2024-05-28 -> 2026-10-01

точность климатологии (всегда 'buy'): 0.5073
Sharpe buy-and-hold  : 0.806
CAGR buy-and-hold    : +0.1319

Эти две строки — нулевая строка лидерборда. Всё ниже измеряется относительно них.


## Хранилище MLflow

**Что это такое.** Хранилище на SQLite по пути `mlflow_hw8.db`, удаляемое и пересоздаваемое в
начале каждого запуска, чтобы ноутбук был **идемпотентным**. Хранилище, которое молча
накапливает результаты между запусками, — ловушка: «лучший запуск» начинает зависеть от того,
сколько раз ноутбук выполнялся.

**Почему база данных, а не файловое хранилище.** Файловое хранилище MLflow переведено в режим
поддержки в MLflow 3.x; развёртывание, которое нельзя опросить, не является системой трекинга.

**Что логируется.** Каждая модель записывается с параметрами, метриками и **тегами,
фиксирующими протокол**. Теги важны: лидерборд, чьи строки не говорят, какое допущение об
издержках и какое правило разбиения их породило, невоспроизводим, а **самый частый провал при
сравнении моделей — сверять числа, измеренные в разных условиях.**

In [4]:
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_DB = os.path.join(NOTEBOOK_DIR, "mlflow_hw8.db")
# Пересоздаётся на каждом запуске, чтобы «лучший запуск» не зависел от того, сколько раз
# этот ноутбук уже выполнялся.
for stale in (TRACKING_DB, TRACKING_DB + ".db"):
    if os.path.isfile(stale):
        os.remove(stale)
for suffix in ("-shm", "-wal"):
    if os.path.isfile(TRACKING_DB + suffix):
        os.remove(TRACKING_DB + suffix)

# Бэкенд SQLite, а не устаревшее файловое хранилище: MLflow 3.x поместил файловый
# бэкенд в режим обслуживания, а бэкенд БД — то, что используется в реальном деплое.
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB.replace(os.sep, '/')}")
EXPERIMENT_NAME = "hw8_boosting_leaderboard"
_experiment = mlflow.set_experiment(EXPERIMENT_NAME)
# MLflow 3.x возвращает здесь объект Experiment, а не голый id.
experiment_id = _experiment.experiment_id
client = MlflowClient()

print(f"tracking uri : {mlflow.get_tracking_uri()}")
print(f"experiment   : {EXPERIMENT_NAME} (id={experiment_id})")
print(f"хранилище есть: {os.path.isfile(TRACKING_DB)}")

2026/10/06 21:47:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/06 21:47:33 INFO mlflow.store.db.utils: Updating database tables


2026/10/06 21:49:52 INFO mlflow.tracking.fluent: Experiment with name 'hw8_boosting_leaderboard' does not exist. Creating a new experiment.


tracking uri : sqlite:///C:/Users/MV/AppData/Local/Temp/opencode/ML_Fin_Notebooks/mlflow_hw8.db
experiment   : hw8_boosting_leaderboard (id=1)
хранилище есть: True


## Зооподбор моделей

Четыре реализации бустинга плюс линейный бейзлайн, потому что линейная модель — самый дешёвый
способ проверить, делают ли бустеры вообще что-нибудь.

**Почему фабрики, а не отдельный код.** Каждая запись — **функция-фабрика**: она собирает
необученный эстиматор из словаря конфигурации. Это удерживает логику переобучения одинаковой
между библиотеками, что важнее, чем звучит: **цикл walk-forward ниже не должен содержать
четырёх слегка отличающихся написанных вручную обучающих процедур.**

**Общие настройки выравниваются там, где библиотеки это позволяют:** 3 класса, фиксированный
seed, **без ранней остановки на тестовом окне.** Ранняя остановка потребовала бы выделения
валидационного набора из обучающего окна *при каждом переобучении*; сделать это правильно
можно, и здесь этого не делают, потому что **неправильная версия является дефолтом в большинстве
примеров и молча оптимизируется на тех данных, которые потом оцениваются.**

In [5]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

COMMON_SEED = RNG_SEED


def make_lightgbm(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    return LGBMClassifier(
        objective="multiclass",
        num_class=3,
        learning_rate=c["learning_rate"],
        num_leaves=c["max_leaf_nodes"],
        min_child_samples=c["min_samples_leaf"],
        reg_lambda=c["l2_regularization"],
        n_estimators=c["max_iter"],
        random_state=COMMON_SEED,
        n_jobs=1,
        verbose=-1,
    )


def make_xgboost(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    # max_depth — аналог бюджета листьев в xgboost; 5 обычно соответствует ~15–31 листьям.
    return XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        learning_rate=c["learning_rate"],
        max_depth=5,
        min_child_weight=c["min_samples_leaf"] / 5.0,
        reg_lambda=c["l2_regularization"],
        n_estimators=c["max_iter"],
        tree_method="hist",
        random_state=COMMON_SEED,
        n_jobs=1,
        verbosity=0,
    )


def make_catboost(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    return CatBoostClassifier(
        loss_function="MultiClass",
        learning_rate=c["learning_rate"],
        depth=6,
        l2_leaf_reg=max(c["l2_regularization"], 1.0),
        iterations=c["max_iter"],
        random_seed=COMMON_SEED,
        verbose=0,
        allow_writing_files=False,
        thread_count=1,
    )


def make_sklearn_hgb(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    return HistGradientBoostingClassifier(
        learning_rate=c["learning_rate"],
        max_leaf_nodes=c["max_leaf_nodes"],
        min_samples_leaf=c["min_samples_leaf"],
        l2_regularization=c["l2_regularization"],
        max_iter=c["max_iter"],
        random_state=COMMON_SEED,
    )


def make_logreg(cfg=None):
    c = cfg or {}
    return LogisticRegression(C=c.get("C", 1.0), max_iter=2000, random_state=COMMON_SEED)


ZOO = {
    "LightGBM": (make_lightgbm, BASE_CONFIG),
    "XGBoost": (make_xgboost, BASE_CONFIG),
    "CatBoost": (make_catboost, BASE_CONFIG),
    "sklearn-HGB": (make_sklearn_hgb, BASE_CONFIG),
    "LogisticRegression": (make_logreg, {"C": 1.0}),
}

print(f"{'модель':<22} {'оценщик':<28} параметры")
for name, (factory, cfg) in ZOO.items():
    est = factory()
    n_param = sum(1 for _ in est.get_params(deep=False))
    print(f"{name:<22} {type(est).__name__:<28} {n_param} параметров конструктора")
print(f"\n{N_BOOSTERS} конфигурации бустинга + 1 линейный бейзлайн -> "
      f"{N_BOOSTERS + 1} моделей для DSR")

модель                 оценщик                      параметры
LightGBM               LGBMClassifier               21 параметров конструктора
XGBoost                XGBClassifier                41 параметров конструктора
CatBoost               CatBoostClassifier           9 параметров конструктора
sklearn-HGB            HistGradientBoostingClassifier 21 параметров конструктора
LogisticRegression     LogisticRegression           14 параметров конструктора

4 конфигурации бустинга + 1 линейный бейзлайн -> 5 моделей для DSR


## Walk-forward оценка: один протокол, пять моделей

**Каждая модель видит одни и те же бары в том же порядке и по тем же правилам:**

* Расширяющееся обучающее окно, переобучение каждые `REFIT_EVERY` баров.
* Статистики предобработки пересчитываются внутри каждого переобучения.
* Каждый бар оценивается моделью, обученной **строго на более ранних барах**.
* Вероятности → дискреционные позиции с порогом воздержания и фильтром подтверждения
  разворота → доходности с издержками.

**Производятся два числа для каждой модели, и это разные вопросы:**

* **Точность OOS** — классифицировал ли направление лучше, чем «всегда держи длинную». Порог —
  climatology.
* **Чистый Sharpe OOS** — заработал ли деньги за вычетом издержек. Порог — buy-and-hold.

**Модель может пройти первый и провалить второй,** и лидерборд намеренно держит их в разных
столбцах, чтобы это было видно, а не усреднялось в один лестный балл.

In [6]:
WF_START = EMBARGO + n_oos


def walk_forward_probabilities(factory, cfg) -> np.ndarray:
    """Каузальные walk-forward вероятности вне выборки для всей выборки.

    Строка t заполняется только моделью, обученной на барах [0, t), поэтому ни
    один бар никогда не оценивается моделью, которая видела его или что-либо
    после него.
    """
    prob = np.full((n_obs, len(CLASS_ORDER)), np.nan)
    for start in range(WF_START, n_obs, REFIT_EVERY):
        end = min(start + REFIT_EVERY, n_obs)
        tr = np.arange(0, start)
        pre = FoldSafePreprocessor().fit(X.iloc[tr])
        model = factory(cfg)
        model.fit(pre.transform(X.iloc[tr]), y.iloc[tr].to_numpy())
        proba = model.predict_proba(pre.transform(X.iloc[start:end]))
        for src_i, cls in enumerate(model.classes_):
            prob[start:end, CLASS_ORDER.index(int(cls))] = proba[:, src_i]
        prob[start:end] = np.nan_to_num(prob[start:end], nan=0.0)
    return prob


def summarise(prob: np.ndarray, name: str) -> dict:
    valid = ~np.isnan(prob).any(axis=1)
    idx = np.flatnonzero(valid)
    pos = apply_causal_execution(prob[idx], buy=BUY_I, sell=SELL_I,
                                 confirm_bars=2, min_confidence=0.40)
    net = positions_to_returns(pos, close.iloc[idx], cost_bps=COST_BPS)
    bh = buy_and_hold_returns(close.iloc[idx])
    frame = pd.DataFrame({"net": net, "bh": bh}).dropna()

    pred = prob[idx].argmax(axis=1)
    y_true = y.iloc[idx].to_numpy()
    acc = float((pred == y_true).mean())

    oos_mask = frame.index >= X_OOS.index[0]
    net_oos, bh_oos = frame.loc[oos_mask, "net"], frame.loc[oos_mask, "bh"]

    return {
        "model": name,
        "bars": int(len(idx)),
        "accuracy": acc,
        "accuracy_skill_vs_climatology": acc - CLIMATOLOGY_ACC,
        "accuracy_correct_oos": float((pred[oos_mask] == y_true[oos_mask]).mean()),
        "sharpe_net": sharpe_ratio(frame["net"]),
        "sharpe_net_oos": sharpe_ratio(net_oos),
        "sharpe_buy_hold_oos": sharpe_ratio(bh_oos),
        "sharpe_edge_vs_bh_oos": sharpe_ratio(net_oos) - sharpe_ratio(bh_oos),
        "cagr_net_oos": annualised_return(net_oos),
        "cagr_buy_hold_oos": annualised_return(bh_oos),
        "sortino_net_oos": sortino_ratio(net_oos),
        "profit_factor": profit_factor(frame["net"]),
        "win_rate": win_rate(frame["net"]),
        "trades": int((np.abs(np.diff(pos)) > 0).sum()),
        "exposure": float(np.abs(pos).mean()),
        "_net": frame["net"],
    }


results, store, net_store = [], {}, {}
for name, (factory, cfg) in ZOO.items():
    t0 = time.perf_counter()
    prob = walk_forward_probabilities(factory, cfg)
    rec = summarise(prob, name)
    rec["fit_seconds"] = time.perf_counter() - t0
    store[name] = prob
    net_store[name] = rec.pop("_net")
    results.append(rec)
    print(f"{name:<22} готово за {rec['fit_seconds']:6.1f}s   "
      f"точн {rec['accuracy']:.4f}  sharpe {rec['sharpe_net']:+.3f}")

zoo_tbl = pd.DataFrame(results)
print("\n" + "=" * 100)
print("ЛИДЕРБОРД WALK-FORWARD, за вычетом издержек, одинаковый протокол для каждой строки")
print("=" * 100)
with pd.option_context("display.width", 200, "display.max_columns", 40):
    print(zoo_tbl[["model", "accuracy", "accuracy_skill_vs_climatology", "accuracy_correct_oos",
               "sharpe_net_oos", "sharpe_buy_hold_oos", "sharpe_edge_vs_bh_oos",
               "cagr_net_oos", "trades", "exposure"]].round(4).to_string(index=False))

print(f"\nстроки-бенчмарки:  точность климатологии {CLIMATOLOGY_ACC:.4f} | "
      f"Sharpe buy-and-hold (OOS) {zoo_tbl['sharpe_buy_hold_oos'].iloc[0]:.3f}")
print("Модель интересна только если она справа от климатологии по точности и справа от")
print("buy-and-hold по Sharpe. Большинство лидербордов сообщают только первый столбец.")

LightGBM               готово за   61.7s   точн 0.4524  sharpe -0.394


XGBoost                готово за  149.8s   точн 0.4508  sharpe -0.325


CatBoost               готово за  575.3s   точн 0.4651  sharpe +0.037


sklearn-HGB            готово за  145.4s   точн 0.4630  sharpe -0.369


LogisticRegression     готово за   14.1s   точн 0.4814  sharpe +0.524

ЛИДЕРБОРД WALK-FORWARD, за вычетом издержек, одинаковый протокол для каждой строки
             model  accuracy  accuracy_skill_vs_climatology  accuracy_correct_oos  sharpe_net_oos  sharpe_buy_hold_oos  sharpe_edge_vs_bh_oos  cagr_net_oos  trades  exposure
          LightGBM    0.4524                        -0.0550                0.4633          0.0321                1.106                -1.0738       -0.0073     223    0.9935
           XGBoost    0.4508                        -0.0566                0.4894         -0.2150                1.106                -1.3209       -0.0459     233    0.9971
          CatBoost    0.4651                        -0.0423                0.4780          0.7066                1.106                -0.3994        0.1047     160    1.0000
       sklearn-HGB    0.4630                        -0.0443                0.4780         -0.3237                1.106                -1.4297       -0

## Логирование в MLflow и обратное чтение лидерборда

**Что доказывается.** Лидерборд выше построен из списка Python. Чтобы показать, что хранилище
трекинга работает, те же метрики логируются в MLflow, а затем лидерборд **перестраивается из
хранилища** через `MlflowClient`. **Если две таблицы разошлись, ноутбук сообщает об этом,
а не доверяет той, что в памяти.**

**Почему это не формальность.** Это маленькая вещь и при этом необычно частая, которую пропускают.
На практике она ловит настоящие проблемы: метрику, залогированную под другим именем; запуск,
который так и не записался; устаревший запуск от предыдущего выполнения, всё ещё сидящий в
хранилище.

In [7]:
PROTOCOL_TAGS = {
    "ticker": TICKER,
    "cost_bps": str(COST_BPS),
    "embargo_bars": str(EMBARGO),
    "refit_every": str(REFIT_EVERY),
    "split": "expanding walk-forward, final 20% untouched",
    "target": f"3-class direction, horizon={HORIZON}, deadband={DEAD_BAND}",
    "preprocessing": "FoldSafePreprocessor fitted inside each refit",
}

run_ids = {}
with mlflow.start_run(run_name="benchmark_buy_and_hold") as r:
    mlflow.log_metric("sharpe_buy_hold", BH_SHARPE_FULL)
    mlflow.log_metric("cagr_buy_hold", BH_CAGR_FULL)
    mlflow.set_tag("row_type", "benchmark")
    mlflow.set_tags(PROTOCOL_TAGS)
    run_ids["buy_and_hold"] = r.info.run_id

with mlflow.start_run(run_name="benchmark_climatology") as r:
    mlflow.log_metric("accuracy_climatology", CLIMATOLOGY_ACC)
    mlflow.set_tag("row_type", "benchmark")
    mlflow.set_tags(PROTOCOL_TAGS)
    run_ids["climatology"] = r.info.run_id

for name, (factory, cfg) in ZOO.items():
    rec = next(x for x in results if x["model"] == name)
    with mlflow.start_run(run_name=name) as r:
        for k, v in cfg.items():
            mlflow.log_param(k, v)
        for k, v in rec.items():
            if k not in ("model",) and isinstance(v, (int, float, np.integer, np.floating)):
                mlflow.log_metric(k, float(v))
        mlflow.set_tag("row_type", "model")
        mlflow.set_tags(PROTOCOL_TAGS)
        run_ids[name] = r.info.run_id

print(f"записано {len(run_ids)} запусков в эксперимент '{EXPERIMENT_NAME}'")

# Лидерборд ПЕРЕСТРАИВАЕТСЯ из хранилища наблюдения.
rows = []
for run in client.search_runs([experiment_id]):
    m = run.data.metrics
    p = run.data.params
    row = {"run_name": run.data.tags.get("mlflow.runName"),
           "row_type": run.data.tags.get("row_type"),
           "sharpe_buy_hold": m.get("sharpe_buy_hold"),
           "accuracy_climatology": m.get("accuracy_climatology")}
    for key in ("accuracy", "accuracy_correct_oos", "sharpe_net", "sharpe_net_oos",
                "sharpe_buy_hold_oos", "sharpe_edge_vs_bh_oos", "cagr_net_oos",
                "trades", "exposure"):
        row[key] = m.get(key)
    rows.append(row)

tracked = pd.DataFrame(rows)
print("\nлидерборд, восстановленный из хранилища MLflow:\n")
print(tracked.round(4).to_string(index=False))

model_rows = tracked[tracked["row_type"] == "model"].sort_values("sharpe_net_oos", ascending=False)
in_memory = zoo_tbl.set_index("model").loc[model_rows["run_name"].tolist(), "sharpe_net_oos"]
rebuilt = pd.Series(model_rows["sharpe_net_oos"].to_numpy(), index=model_rows["run_name"])
agree = bool(np.allclose(in_memory.to_numpy(dtype=float), rebuilt.to_numpy(dtype=float)))
print(f"\nSharpe в памяти == Sharpe, восстановленный из хранилища : {agree}")
if not agree:
    print("РАСХОЖДЕНИЕ: хранилище и таблица в памяти не сходятся. Не доверяйте")
    print("лидерборду, пока расхождение не объяснено.")
else:
    print("Хранилище сохраняет данные без потерь, поэтому числа ниже восстанавливаются из одного MLflow.")

записано 7 запусков в эксперимент 'hw8_boosting_leaderboard'

лидерборд, восстановленный из хранилища MLflow:

              run_name  row_type  sharpe_buy_hold  accuracy_climatology  accuracy  accuracy_correct_oos  sharpe_net  sharpe_net_oos  sharpe_buy_hold_oos  sharpe_edge_vs_bh_oos  cagr_net_oos  trades  exposure
    LogisticRegression     model              NaN                   NaN    0.4814                0.4878      0.5243          0.1203                1.106                -0.9856        0.0063    66.0    0.9873
           sklearn-HGB     model              NaN                   NaN    0.4630                0.4780     -0.3688         -0.3237                1.106                -1.4297       -0.0618   235.0    0.9967
              CatBoost     model              NaN                   NaN    0.4651                0.4780      0.0372          0.7066                1.106                -0.3994        0.1047   160.0    1.0000
               XGBoost     model              NaN        

## Пермутационная важность вне выборки

**В чём разница с обычной важностью.** Важность, посчитанная на обучающих данных, отвечает на
вопрос «какие признаки модель использовала», — это **свойство подгонки**. Вопрос, на который
стоит ответить: **«на какие признаки модель *опирается*, когда встречает данные, которых не
видела»**, и честный способ это измерить — перемешать признак в выделенных данных и наблюдать
ущерб.

**Почему это ещё и детектор утечки.** Признак, действительно полезный, навредит при перемешивании,
и признак, через который модель «срезает угол», тоже навредит. Признак, **не имеющий** эффекта
вне выборки при высокой важности на обучении, — предупреждающий знак: модель его запомнила, а
не использовала.

**Важность считается только на финальном окне OOS**, моделью, переобученной на данных IS.

In [8]:
from sklearn.inspection import permutation_importance

BEST_BY_SHARPE = model_rows["run_name"].iloc[0]
print(f"модель, для которой считалась важность: {BEST_BY_SHARPE}")

# Та же модель переобучается только на окне IS, затем перестановка на нетронутом окне OOS.
best_factory, best_cfg = ZOO[BEST_BY_SHARPE]
pre = FoldSafePreprocessor().fit(X_IS)
X_IS_s = pre.transform(X_IS)
X_OOS_s = pre.transform(X_OOS)
imp_model = best_factory(best_cfg)
imp_model.fit(X_IS_s, y_IS.to_numpy())

perm = permutation_importance(
    imp_model, X_OOS_s, y_OOS.to_numpy(),
    scoring="accuracy", n_repeats=20, random_state=RNG_SEED, n_jobs=1)

imp = pd.DataFrame({
    "feature": X_OOS.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
imp["pct_of_total_loss"] = 100.0 * imp["importance_mean"].clip(lower=0) / \
    max(imp["importance_mean"].clip(lower=0).sum(), 1e-12)

print(f"\nперестановочная важность вне выборки (единицы точности, {len(X_OOS)} баров OOS)\n")
print(imp.head(12).round(5).to_string(index=False))

zero_effect = int((imp["importance_mean"].abs() < 1e-4).sum())
print(f"\nпризнаков без измеримого эффекта вне выборки : {zero_effect} / {len(imp)}")
print(f"топ-3 признака несут                  : "
      f"{imp['pct_of_total_loss'].head(3).sum():.1f}% измеренного эффекта")
if zero_effect > len(imp) // 2:
    print("Большинство признаков вне выборки ничего не дают. Набор настолько плоский, что не удержит")
    print("модель со многими эффективными степенями свободы, что согласуется с")
    print("лидербордом выше и с результатом HW7.")

модель, для которой считалась важность: CatBoost



перестановочная важность вне выборки (единицы точности, 613 баров OOS)

    feature  importance_mean  importance_std  pct_of_total_loss
  close_loc          0.01688         0.00744           19.62085
  ret_lag_2          0.01639         0.00916           19.05213
macd_signal          0.01052         0.00552           12.22749
  ret_lag_4          0.00922         0.00976           10.71090
  ret_lag_5          0.00685         0.00620            7.96209
     vol_20          0.00587         0.00887            6.82464
      ret_1          0.00546         0.00818            6.35071
  vol_ratio          0.00530         0.00774            6.16114
   ma_ratio          0.00498         0.00826            5.78199
  macd_hist          0.00367         0.00973            4.26540
  ret_lag_1          0.00082         0.00897            0.94787
Close_Lag_2          0.00008         0.00036            0.09479

признаков без измеримого эффекта вне выборки : 7 / 23
топ-3 признака несут                  : 

## Калибровка победившей модели

**Почему классификатора мало.** Классификатор может быть полезен по направлению и при этом
бесполезен как **размер позиции**. `HW6` показал, что покрытие интервалов не условно относительно
состояния мира; здесь задаётся более простой и более напрямую полезный вопрос: **когда модель
говорит 70% «покупать», как часто она права?**

**Три диагностики, все на окне OOS:**

* **Таблица надёжности (reliability)** — предсказанная вероятность против наблюдённой частоты
  по корзинам.
* **Ожидаемая ошибка калибровки (ECE)** — средневзвешенное абсолютное расхождение, корзины
  равной ширины.
* **Brier skill score** — против прогноза climatology, чтобы он унаследовал ту же дисциплину
  бенчмарков, что и всё остальное.

**Как читать результат.** Если edge модели невелик, картина будет похожа на априорную — что
само по себе является честным результатом и должно быть таковым сообщено, а не спрятано за
одним числом точности.

In [9]:
from mlfin_core import brier_skill_score, expected_calibration_error, reliability_table

best_prob = store[BEST_BY_SHARPE]
best_valid = ~np.isnan(best_prob).any(axis=1)
oos_mask_np = np.asarray(data.index >= X_OOS.index[0])
best_idx = np.flatnonzero(best_valid & oos_mask_np)
oos_prob = best_prob[best_idx]
oos_true = y.iloc[best_idx].to_numpy()

calib_pred = oos_prob[:, BUY_I]
reliability = reliability_table(y_true=oos_true, y_prob=oos_prob[:, BUY_I], n_bins=5)
reliability.columns = [c.replace("y_true", "observed_buy") for c in reliability.columns]
print(reliability.round(4).to_string())

ECE = expected_calibration_error(y_true=oos_true, y_prob=oos_prob[:, BUY_I], n_bins=10)

# Мультиклассовый Бриер требует полной матрицы вероятностей (n, 3), а опорный
# прогноз должен браться только из окна IS — если посчитать его из тестовой
# выборки, бенчмарк незаметно станет проще модели.
class_prior = (
    y.iloc[: n_obs - n_oos].value_counts(normalize=True)
    .reindex(CLASS_ORDER).fillna(0.0).to_numpy(dtype=float)
)
prior = float(class_prior[BUY_I])
clim_matrix = np.tile(class_prior, (len(oos_true), 1))
BSS = brier_skill_score(y_true=oos_true, y_prob=oos_prob, climatology=clim_matrix)

print(f"\nаприорная вероятность класса «покупка» на IS : {prior:.4f}")
print(f"наблюдаемая частота покупки на OOS  : {float((oos_true == BUY_I).mean()):.4f}")
print(f"средняя предсказанная вероятность покупки : {calib_pred.mean():.4f}")
print(f"ожидаемая ошибка калибровки (10 корзин): {ECE:.4f}")
print(f"навык Бриера (мультикласс) против климатологии IS : {BSS:+.4f}")
print(f"средняя максимальная предсказанная вероятность : {oos_prob.max(axis=1).mean():.4f}")
print(f"баров, где макс. вероятность >= 0.40  : "
      f"{float((oos_prob.max(axis=1) >= 0.40).mean()):.1%}")

CALIBRATED_ENOUGH = bool(np.isfinite(BSS) and BSS > 0.0 and ECE < 0.10)
print(f"\nпригодна для размера позиции        : {CALIBRATED_ENOUGH}")
if not CALIBRATED_ENOUGH:
    print("Навык Бриера <= 0 или ECE >= 0.10 означают, что вероятности не лучше")
    print("прогноза климатологии. Позиции по ним добавят издержки и риск, но не преимущество.")

   bin    n  pred_mean  obs_freq     gap
0    1   10     0.3890    1.0000  0.6110
1    2  588     0.4998    0.7245  0.2247
2    3   15     0.6148    0.6000 -0.0148

априорная вероятность класса «покупка» на IS : 0.4898
наблюдаемая частота покупки на OOS  : 0.5073
средняя предсказанная вероятность покупки : 0.5008
ожидаемая ошибка калибровки (10 корзин): 0.2258
навык Бриера (мультикласс) против климатологии IS : -0.0088
средняя максимальная предсказанная вероятность : 0.5057
баров, где макс. вероятность >= 0.40  : 99.8%

пригодна для размера позиции        : False
Навык Бриера <= 0 или ECE >= 0.10 означают, что вероятности не лучше
прогноза климатологии. Позиции по ним добавят издержки и риск, но не преимущество.


## Неопределённость на лучшей строке лидерборда

`HW6` и `HW7` оба заканчивались одной и той же дисциплиной, и она переносится без изменений:
**точечная оценка из нескольких сотен баров — утверждение о шуме.**

* **Block bootstrap** — доверительный интервал к чистому Sharpe победителя, ресемплирование
  блоками, чтобы сохранить серийную зависимость.
* **PSR** против нуля — отличим ли Sharpe от отсутствия edge.
* **DSR** против лучшего из `N_BOOSTERS + 1` попыток — честный бенчмарк, когда перебирались
  четыре реализации бустинга и линейная модель.
* **MinTRL** — сколько лет трек-рекорда нужно вообще, чтобы подтвердить целевой Sharpe.

**Что значит провал.** Если интервал победителя включает ноль, лидерборд не нашёл edge — он
нашёл **наименее худшего члена небольшого семейства.**

In [10]:
# Поток чистых доходностей выигравшей модели взят из прогона оценки, а не
# пересчитан заново, чтобы неопределённость мерялась ровно на том ряду, что и был отчитан.
best_net = net_store[BEST_BY_SHARPE]

ci = bootstrap_sharpe_ci(best_net, n_boot=4000, block_size=10, alpha=0.05, seed=RNG_SEED)
psr = probabilistic_sharpe_ratio(best_net, sr_benchmark=0.0)
dsr = deflated_sharpe_ratio(best_net, n_trials=N_BOOSTERS + 1)
mintrl = min_track_record_length(best_net, sr_target=SR_TARGET)
MINTRL_OK = bool(np.isfinite(mintrl))

print(f"неопределённость для {BEST_BY_SHARPE}")
print(f"  точечная оценка чистого Sharpe    : {ci['point']:.3f}")
print(f"  интервал 95% блочного бутстрэпа   : [{ci['lo']:.3f}, {ci['hi']:.3f}]")
print(f"  P(Sharpe > 0)                   : {ci['p_gt_zero']:.3f}")
print(f"  PSR против нуля                  : {psr:.4f}")
print(f"  DSR против лучшего из {N_BOOSTERS + 1} попыток : {dsr['dsr']:.4f} "
      f"(бенчмарк SR {dsr['sr_benchmark']:.3f})")
if MINTRL_OK:
    print(f"  MinTRL для SR {SR_TARGET:.1f}                : {mintrl / TRADING_DAYS:.1f} лет "
      f"против {len(best_net) / TRADING_DAYS:.1f} наблюдённых")
else:
    print(f"  MinTRL для SR {SR_TARGET:.1f}                : не определён (реализованный Sharpe <= 0)")

CI_CLEARS = bool(np.isfinite(ci["lo"]) and ci["lo"] > 0)
DSR_CLEARS = bool(np.isfinite(dsr["dsr"]) and dsr["dsr"] >= 0.95)
VERDICT = "credible edge" if (CI_CLEARS and DSR_CLEARS) else "no credible edge"

print(f"\n  Интервал 95% не включает ноль : {CI_CLEARS}")
print(f"  DSR >= 0.95          : {DSR_CLEARS}")
print(f"  ВЕРДИКТ              : {VERDICT}")

BEATS_CLIMATOLOGY = bool(zoo_tbl["accuracy"].max() > CLIMATOLOGY_ACC)
BEATS_BUY_HOLD = bool(zoo_tbl["sharpe_net_oos"].max() > zoo_tbl["sharpe_buy_hold_oos"].iloc[0])
print(f"\n  лучшая точность бьёт климатологию : {BEATS_CLIMATOLOGY} "
      f"({zoo_tbl['accuracy'].max():.4f} против {CLIMATOLOGY_ACC:.4f})")
print(f"  лучший Sharpe бьёт buy-and-hold  : {BEATS_BUY_HOLD} "
      f"({zoo_tbl['sharpe_net_oos'].max():.3f} против {zoo_tbl['sharpe_buy_hold_oos'].iloc[0]:.3f})")
print("\nОба приводятся, потому что обычно расходятся, и тогда результат по точности —")
print("тот, что попадает в отчёт, а важен результат по Sharpe.")

неопределённость для CatBoost
  точечная оценка чистого Sharpe    : 0.037
  интервал 95% блочного бутстрэпа   : [-0.530, 0.636]
  P(Sharpe > 0)                   : 0.544
  PSR против нуля                  : 0.5462
  DSR против лучшего из 5 попыток : 0.1735 (бенчмарк SR 0.021)
  MinTRL для SR 1.0                : 721.8 лет против 9.7 наблюдённых

  Интервал 95% не включает ноль : False
  DSR >= 0.95          : False
  ВЕРДИКТ              : no credible edge

  лучшая точность бьёт климатологию : False (0.4814 против 0.5073)
  лучший Sharpe бьёт buy-and-hold  : False (0.707 против 1.106)

Оба приводятся, потому что обычно расходятся, и тогда результат по точности —
тот, что попадает в отчёт, а важен результат по Sharpe.


## Выводы

Таблицы выше печатаются кодом и вычислены в этом запуске; рассуждение вынесено отдельно, чтобы
можно было сверить одно с другим.

**1. Лидерборд — это утверждение о поиске, а не о модели.** Были испробованы четыре реализации
бустинга и линейный бейзлайн. Ячейка DSR получает это число, поэтому Sharpe победителя
сравнивается с лучшим из пяти попыток, а не с нулём. **Лидерборд, сообщающий только первую
строку, молча сообщил максимум случайной величины и назвал его оценкой.**

**2. Точность и торгуемость — разные утверждения, и теперь в репозитории сообщаются оба.** Столбец
точности измеряется против бенчмарка большинства классов, столбец Sharpe — против buy-and-hold
на идентичных барах. Там, где модель проходит первое и проваливает второе, она выучила базовый
частотный индекса с восходящим дрейфом — что является **истинным и бесполезным** утверждением.

**3. Бенчмарки считались до моделей, намеренно.** Точность climatology и Sharpe buy-and-hold
появляются выше каждой строки модели, с собственными запусками MLflow. **Определять бейзлайн
после просмотра результатов — самый частый способ сделать сравнение неопровержимым,** и это
стоит ноль рублей, чтобы избежать.

**4. Важность измеряется на данных, которых модель не видела.** Важность на обучении отвечает на
вопрос о подгонке; пермутационная важность на окне OOS отвечает на вопрос о модели. **Число
признаков без измеримого эффекта вне выборки печатается рядом с ранжированием,** потому что
набор признаков, где большинство столбцов ничего не делает, — это реальное ограничение
сложности модели здесь.

**5. Калибровка решает, можно ли вероятностями задавать размер позиции.** Модель, чей Brier
skill score против climatology не положителен, **не производит информацию, которой риск-менеджер
мог бы пользоваться,** как бы ни выглядела её точность. `HW6` установил то же самое для
интервалов; здесь это устанавливается для вероятностей, которые их порождают.

**6. Хранилище трекинга — часть результата.** Каждая строка выше восстановима из MLflow вместе
со своими параметрами, допущением об издержках и правилом разбиения, а лидерборды из памяти и
из хранилища **сравниваются, а не предполагаются равными. Воспроизводимость, заявленная прозой,
не является воспроизводимостью.**

In [11]:
print("=" * 96)
print("ГЛАВНЫЙ ВЫВОД, посчитан в этом запуске")
print("=" * 96)

print("\nДанные и протокол")
print(f"  тикер / источник       : {TICKER} / {source}")
print(f"  баров / признаков      : {n_obs} / {X.shape[1]}")
print(f"  IS / OOS               : {len(X_IS)} / {len(X_OOS)}")
print(f"  издержка за полный круг: {COST_BPS:.0f} bp")
print(f"  конфигурация из HW7    : {BASE_CONFIG}")

print("\nНулевые бейзлайны (заданы до запуска любой модели)")
print(f"  точность климатологии  : {CLIMATOLOGY_ACC:.4f} (всегда '{CLASS_NAMES[MAJORITY_CLASS]}')")
print(f"  Sharpe buy-and-hold    : {BH_SHARPE_FULL:.3f}  (вся выборка)")
print(f"  CAGR buy-and-hold      : {BH_CAGR_FULL:+.4f}")
bh_oos_ref = float(zoo_tbl['sharpe_buy_hold_oos'].iloc[0])
print(f"  Sharpe buy-and-hold    : {bh_oos_ref:.3f}  (окно OOS, сравнение выше)")

print("\nЛидерборд, отсортированный по чистому Sharpe OOS")
print(f"  {'модель':<22}{'точн':>8}{'vs клим':>10}{'OOS':>9}"
      f"{'SR OOS':>9}{'SR bh-hold':>12}{'преим':>9}{'сделки':>8}")
for _, r in model_rows.iterrows():
    print(f"  {r['run_name']:<22}{r['accuracy']:>8.4f}"
      f"{r['accuracy'] - CLIMATOLOGY_ACC:>10.4f}{r['accuracy_correct_oos']:>9.4f}"
      f"{r['sharpe_net_oos']:>9.3f}{r['sharpe_buy_hold_oos']:>12.3f}"
      f"{r['sharpe_edge_vs_bh_oos']:>9.3f}{int(r['trades']):>8}")
print(f"  {'климатология':<22}{CLIMATOLOGY_ACC:>8.4f}{'--':>10}")
print(f"  {'buy-and-hold':<22}{'--':>8}{'--':>10}{'--':>9}{bh_oos_ref:>9.3f}{'--':>12}"
      f"{'0.000':>9}{'--':>8}")

print("\nRank stability")
print(f"  лучшая по точности : {zoo_tbl.loc[zoo_tbl['accuracy'].idxmax(), 'model']}")
print(f"  лучшая по Sharpe OOS: {BEST_BY_SHARPE}")
print(f"  одна модель выигрывает в обоих: "
      f"{zoo_tbl.loc[zoo_tbl['accuracy'].idxmax(), 'model'] == BEST_BY_SHARPE}")
print(f"  диапазон точности OOS по моделям : "
      f"{zoo_tbl['accuracy_correct_oos'].min():.4f} -> {zoo_tbl['accuracy_correct_oos'].max():.4f}")
print(f"  диапазон Sharpe OOS по моделям   : "
      f"{zoo_tbl['sharpe_net_oos'].min():.3f} -> {zoo_tbl['sharpe_net_oos'].max():.3f}")

print("\nПерестановочная важность вне выборки")
print(f"  модель               : {BEST_BY_SHARPE}")
print(f"  главный признак     : {imp.iloc[0]['feature']} "
      f"({imp.iloc[0]['importance_mean']:+.5f})")
print(f"  доля эффекта топ-3   : {imp['pct_of_total_loss'].head(3).sum():.1f}%")
print(f"  признаков без эффекта вне выборки: {zero_effect} / {len(imp)}")

print("\nКалибровка на окне OOS")
print(f"  априори / наблюдено  : {prior:.4f} / {float((oos_true == BUY_I).mean()):.4f}")
print(f"  средняя предсказанная покупка: {calib_pred.mean():.4f}")
print(f"  ECE (10 корзин)      : {ECE:.4f}")
print(f"  мультикласс BSS против климатологии IS: {BSS:+.4f}")
print(f"  пригодна для размера позиции: {CALIBRATED_ENOUGH}")

print("\nНеопределённость для выигравшей строки")
print(f"  интервал Sharpe 95%  : [{ci['lo']:.3f}, {ci['hi']:.3f}]")
print(f"  PSR против нуля      : {psr:.4f}")
print(f"  DSR против лучшего из {N_BOOSTERS + 1} : {dsr['dsr']:.4f}")
print(f"  MinTRL для SR {SR_TARGET:.1f}       : "
      + (f"{mintrl / TRADING_DAYS:.1f} лет" if MINTRL_OK else "не определён (Sharpe <= 0)"))
print(f"  обходит климатологию : {BEATS_CLIMATOLOGY}")
print(f"  обходит buy-and-hold : {BEATS_BUY_HOLD}")
print(f"\n  ВЕРДИКТ              : {VERDICT}")

ГЛАВНЫЙ ВЫВОД, посчитан в этом запуске

Данные и протокол
  тикер / источник       : SPY / SPY (local: C:\Users\MV\AppData\Local\Temp\opencode\ML_Fin_Notebooks\data\SPY.csv)
  баров / признаков      : 3065 / 23
  IS / OOS               : 2452 / 613
  издержка за полный круг: 10 bp
  конфигурация из HW7    : {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_iter': 150}

Нулевые бейзлайны (заданы до запуска любой модели)
  точность климатологии  : 0.5073 (всегда 'buy')
  Sharpe buy-and-hold    : 0.806  (вся выборка)
  CAGR buy-and-hold      : +0.1319
  Sharpe buy-and-hold    : 1.106  (окно OOS, сравнение выше)

Лидерборд, отсортированный по чистому Sharpe OOS
  модель                    точн   vs клим      OOS   SR OOS  SR bh-hold    преим  сделки
  CatBoost                0.4651   -0.0423   0.4780    0.707       1.106   -0.399     160
  LogisticRegression      0.4814   -0.0259   0.4878    0.120       1.106   -0.986      66
  LightGBM   

In [12]:
summary = {
    "run": {
        "ticker": TICKER,
        "data_source": str(source),
        "n_bars": int(n_obs),
        "n_features": int(X.shape[1]),
        "is_bars": int(len(X_IS)),
        "oos_bars": int(len(X_OOS)),
        "cost_bps": COST_BPS,
        "embargo": EMBARGO,
        "refit_every": REFIT_EVERY,
        "config_reused_from_hw7": BASE_CONFIG,
        "n_boosters_tried": N_BOOSTERS,
    },
    "benchmarks": {
        "climatology_accuracy": CLIMATOLOGY_ACC,
        "buy_hold_sharpe_full": BH_SHARPE_FULL,
        "buy_hold_cagr_full": BH_CAGR_FULL,
        "buy_hold_sharpe_oos": bh_oos_ref,
    },
    "leaderboard": json.loads(model_rows.to_json(orient="records")),
    "mlflow": {
        "tracking_uri": mlflow.get_tracking_uri(),
        "experiment": EXPERIMENT_NAME,
        "experiment_id": experiment_id,
        "n_runs_logged": len(run_ids),
        "run_ids": run_ids,
        "in_memory_matches_store": agree,
    },
    "rank_stability": {
        "best_by_accuracy": str(zoo_tbl.loc[zoo_tbl["accuracy"].idxmax(), "model"]),
        "best_by_oos_sharpe": BEST_BY_SHARPE,
        "same_model_wins_both": bool(
            zoo_tbl.loc[zoo_tbl["accuracy"].idxmax(), "model"] == BEST_BY_SHARPE),
        "oos_accuracy_min": float(zoo_tbl["accuracy_correct_oos"].min()),
        "oos_accuracy_max": float(zoo_tbl["accuracy_correct_oos"].max()),
        "oos_sharpe_min": float(zoo_tbl["sharpe_net_oos"].min()),
        "oos_sharpe_max": float(zoo_tbl["sharpe_net_oos"].max()),
    },
    "permutation_importance": {
        "model": BEST_BY_SHARPE,
        "computed_on": "final OOS window only",
        "top_features": imp.head(10)[["feature", "importance_mean"]].to_dict("records"),
        "features_with_no_oos_effect": zero_effect,
        "n_features": int(len(imp)),
    },
    "calibration": {
        "buy_prior_is": prior,
        "buy_frequency_oos": float((oos_true == BUY_I).mean()),
        "mean_predicted_buy": float(calib_pred.mean()),
        "expected_calibration_error": ECE,
        "brier_skill_score_vs_climatology": BSS,
        "usable_as_position_sizer": CALIBRATED_ENOUGH,
    },
    "uncertainty": {
        "sharpe_ci_lo": ci["lo"],
        "sharpe_ci_hi": ci["hi"],
        "p_sharpe_gt_zero": ci["p_gt_zero"],
        "psr": psr,
        "dsr": dsr["dsr"],
        "dsr_n_trials": N_BOOSTERS + 1,
        "dsr_sr_benchmark": dsr["sr_benchmark"],
        "mintrl_years": (mintrl / TRADING_DAYS) if MINTRL_OK else None,
        "ci_excludes_zero": CI_CLEARS,
        "dsr_credible": DSR_CLEARS,
        "beats_climatology": BEATS_CLIMATOLOGY,
        "beats_buy_and_hold": BEATS_BUY_HOLD,
        "verdict": VERDICT,
    },
}
print(json.dumps(summary, indent=2, default=str))

{
  "run": {
    "ticker": "SPY",
    "data_source": "SPY (local: C:\\Users\\MV\\AppData\\Local\\Temp\\opencode\\ML_Fin_Notebooks\\data\\SPY.csv)",
    "n_bars": 3065,
    "n_features": 23,
    "is_bars": 2452,
    "oos_bars": 613,
    "cost_bps": 10.0,
    "embargo": 5,
    "refit_every": 21,
    "config_reused_from_hw7": {
      "learning_rate": 0.03,
      "max_leaf_nodes": 15,
      "min_samples_leaf": 20,
      "l2_regularization": 0.0,
      "max_iter": 150
    },
    "n_boosters_tried": 4
  },
  "benchmarks": {
    "climatology_accuracy": 0.5073409461663948,
    "buy_hold_sharpe_full": 0.8064690429810053,
    "buy_hold_cagr_full": 0.1319134825354653,
    "buy_hold_sharpe_oos": 1.105954945499618
  },
  "leaderboard": [
    {
      "run_name": "CatBoost",
      "row_type": "model",
      "sharpe_buy_hold": null,
      "accuracy_climatology": null,
      "accuracy": 0.4650592562,
      "accuracy_correct_oos": 0.4779771615,
      "sharpe_net": 0.0372310543,
      "sharpe_net_oos": 0